#### Problem Statement:
We have an employee-device dataset containing employee_id, device_type (Laptop, Tab, Mobile), issued_date, and status (Active/Inactive). The goal is to find employees with only a Laptop in an Active state and no other devices.

🔹 What will you learn?
✅ How to filter and exclude records efficiently in PySpark

✅ Using filter(), isin(), join(), and left_anti in PySpark

✅ Practical Data Engineering interview question


In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Spark-Q003")
    .master("local[*]").config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/27 10:43:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


http://vmware-kubuntu.sandbox.net:4040


In [1]:
data = [
    (101, "Laptop", "2024-01-15", "Active"),
    (101, "Mobile", "2024-02-10", "Active"),
    (102, "Laptop", "2024-03-05", "Active"),
    (103, "Laptop", "2023-12-20", "Inactive"),
    (103, "Tab", "2024-02-25", "Active"),
    (104, "Mobile", "2024-04-10", "Active"),
    (104, "Laptop", "2024-01-30", "Inactive"),
    (105, "Laptop", "2024-03-15", "Active"),
    (105, "Tab", "2024-02-05", "Inactive"),
    (105, "Mobile", "2024-01-20", "Inactive"),
]

columns = ["employee_id", "device_type", "issued_date", "status"]
df = spark.createDataFrame(data, columns)

df.show()



+-----------+-----------+-----------+--------+
|employee_id|device_type|issued_date|  status|
+-----------+-----------+-----------+--------+
|        101|     Laptop| 2024-01-15|  Active|
|        101|     Mobile| 2024-02-10|  Active|
|        102|     Laptop| 2024-03-05|  Active|
|        103|     Laptop| 2023-12-20|Inactive|
|        103|        Tab| 2024-02-25|  Active|
|        104|     Mobile| 2024-04-10|  Active|
|        104|     Laptop| 2024-01-30|Inactive|
|        105|     Laptop| 2024-03-15|  Active|
|        105|        Tab| 2024-02-05|Inactive|
|        105|     Mobile| 2024-01-20|Inactive|
+-----------+-----------+-----------+--------+



In [5]:
from pyspark.sql.functions import *
from pyspark.sql.window import *

windowSpec = Window.partitionBy("employee_id")

result_df = df.withColumn("device_count", count("device_type").over(windowSpec))

result_df.filter((col("device_count") == 1) & (col("status") == "Active")).show()


+-----------+-----------+-----------+------+------------+
|employee_id|device_type|issued_date|status|device_count|
+-----------+-----------+-----------+------+------------+
|        102|     Laptop| 2024-03-05|Active|           1|
+-----------+-----------+-----------+------+------------+



Spark SQL Solution






In [ ]:
%sql

df.createOrReplaceTempView("device_data")

SELECT employee_id FROM device_data
WHERE device_type = 'Laptop' AND status = 'Active'
AND employee_id NOT IN (
    SELECT employee_id 
    FROM device_data 
    WHERE device_type IN ('Tab', 'Mobile') AND status = 'Active'
)

#### Approach - 2

##### Step 1: Get employees who have an active Laptop

In [6]:
active_laptop_df = df.filter((col("device_type") == "Laptop") & (col("status") == "Active")).select("employee_id")
active_laptop_df.show()

+-----------+
|employee_id|
+-----------+
|        101|
|        102|
|        105|
+-----------+



##### Step 2: Get employees who have any other device (Tab/Mobile)

In [8]:
other_devices_df = df.filter(col("device_type").isin(["Tab", "Mobile"]) & (col("status") == "Active") ).select("employee_id").distinct()
other_devices_df.show()

+-----------+
|employee_id|
+-----------+
|        101|
|        103|
|        104|
+-----------+



##### Step 3: Exclude employees who have other devices

In [10]:
only_active_laptop_df = active_laptop_df.join(other_devices_df, "employee_id", "left_anti")
only_active_laptop_df.show()

+-----------+
|employee_id|
+-----------+
|        102|
|        105|
+-----------+

